In [ ]:
!pip install hydrafloods
!pip install geopandas
import pandas as pd
import ee
import hydrafloods as hf
from hydrafloods import corrections
import logging
import concurrent.futures
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

# Initialize Earth Engine only once at the beginning
print("Initializing Google Earth Engine...")
ee.Authenticate()
ee.Initialize(project='ee-ageidv')  # Initialize with your project
print("Earth Engine initialized successfully.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.9/73.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.6/86.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.2/463.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.3/235.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.5/213.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.0/349.0 kB 4.4 MB/s eta 0:00:

In [ ]:
import glob
import logging
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from functools import reduce
from multiprocessing import Manager, Pool, Value, cpu_count
from typing import Any, Dict, List, Optional

import ee
import pandas as pd
import yaml
from tqdm import tqdm


class TeeLogger(object):
    def __init__(self, filename):
        self.terminal = sys.stdout
        self.log = open(filename, "a", encoding="utf-8")

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.flush()

    def flush(self):
        self.terminal.flush()
        self.log.flush()


def setup_logger(log_file_path: str) -> logging.Logger:
    """Set up enhanced logging to both file and console."""
    logging.root.setLevel(logging.INFO)

    # Create handlers
    file_handler = logging.FileHandler(log_file_path)
    stream_handler = logging.StreamHandler(sys.stdout)

    # Create formatter
    formatter = logging.Formatter(
        "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
    )
    file_handler.setFormatter(formatter)
    stream_handler.setFormatter(formatter)

    # Get logger and add handlers
    logger = logging.getLogger()
    logger.addHandler(file_handler)
    logger.addHandler(stream_handler)

    # Redirect stdout
    sys.stdout = TeeLogger(log_file_path)

    logger.info(f"Logger initialized. Output will be saved at: {log_file_path}")
    return logger


def load_config(config_path: str) -> dict:
    """Load and validate configuration from YAML file."""
    logger.info(f"Loading configuration from: {config_path}")
    with open(config_path, "r") as config_file:
        config = yaml.safe_load(config_file)

    required_keys = [
        "project_id",
        "output_path",
        "shapefile_path",
        "id_field",
        "start_date",
        "end_date",
        "max_workers",
        "batch_size",
    ]

    missing_keys = [key for key in required_keys if key not in config]
    if missing_keys:
        raise ValueError(f"Missing required configuration keys: {missing_keys}")

    logger.info("Configuration loaded successfully")
    return config


# This makes the logger object available globally
logger = setup_logger("/content/drive/MyDrive/SAR_rgl/output_2/modis_processing.log")


def setup_environment():
    print("Setting up processing environment...")

    print("Loading shapefile...")
    shapefile = ee.FeatureCollection(config["shapefile_path"])
    idname = config["id_field"]
    print(f"Shapefile loaded. Using ID field: {idname}")

    # Porto Alegre area bounds (updated to match v5 grid)
    BOUNDS = {
        "minx": -51.65,  # West
        "miny": -28.45,  # South
        "maxx": -51.25,  # East
        "maxy": -28.05   # North
    }
    bbox = ee.Geometry.Rectangle(
        [BOUNDS["minx"], BOUNDS["miny"], BOUNDS["maxx"], BOUNDS["maxy"]]  # minx  # miny  # maxx 2.7  # maxy 1.2
    )

    # Filter the shapefile to include only features that intersect with the bounding box
    filtered_shapefile = shapefile.filterBounds(bbox)

    print("Getting list of polygon IDs...")
    ids = filtered_shapefile.aggregate_array(idname).getInfo()
    print(f"Number of polygons after filtering: {len(ids)}")

    print("Defining elevation datasets...")
    elv = ee.Image("JAXA/ALOS/AW3D30/V2_2").select("AVE_DSM")
    merit = ee.Image("MERIT/Hydro/v1_0_1")
    dem = merit.select("elv").unmask(0)
    hand = merit.select("hnd").unmask(0)
    print("Elevation datasets defined.")

    print("Environment setup completed.")
    return filtered_shapefile, ids, dem, hand


def fc_to_dataframe(fc):
    """Convert Earth Engine FeatureCollection to pandas DataFrame with error handling."""
    try:
        # Get features with error handling
        features = fc.getInfo()
        if not features or "features" not in features:
            logger.warning("No features found in FeatureCollection")
            return pd.DataFrame()

        features = features["features"]
        if not features:
            logger.warning("Empty features list in FeatureCollection")
            return pd.DataFrame()

        # Extract properties
        data = []
        for feature in features:
            if "properties" in feature and feature["properties"]:
                properties = feature["properties"]
                properties["index"] = feature.get("id")
                data.append(properties)

        # Create DataFrame
        if not data:
            logger.warning("No valid properties found in features")
            return pd.DataFrame()

        df = pd.DataFrame(data)
        return df

    except Exception as e:
        logger.error(f"Error converting FeatureCollection to DataFrame: {e}")
        return pd.DataFrame()


# Original MODIS processing functions with added logging
def dfo_bands_gq(collection):
    """Rename bands in MODIS GQ collections."""
    logger.debug("Renaming MODIS GQ bands")
    return collection.select(
        ["sur_refl_b01", "sur_refl_b02", "num_observations"],
        ["red_250m", "nir_250m", "obs250"],
    )


# Function that renames the bands in MODIS GA (500-m) collections to readable band names
def dfo_bands_ga(collection):
    return collection.select(
        [
            "sur_refl_b01",
            "sur_refl_b02",
            "sur_refl_b03",
            "sur_refl_b04",
            "sur_refl_b05",
            "sur_refl_b06",
            "sur_refl_b07",
            "state_1km",
            "num_observations_500m",
        ],
        [
            "red_500m",
            "nir_500m",
            "blue",
            "green",
            "band5",
            "band6",
            "swir",
            "state_1km",
            "obs500",
        ],
    )


# Join_collections joins the GQ & GA products base on similar time-fields.  The bands of each image are concatenated together.  The results is several bands with different resolutions!
def join_collections(img_coll1, img_coll2):
    filter_time_eq = ee.Filter.equals(
        leftField="system:time_start", rightField="system:time_start"
    )
    joined = ee.Join.inner().apply(img_coll1, img_coll2, filter_time_eq)

    def image_cat(image):
        return ee.Image.cat(image.get("primary"), image.get("secondary"))

    return joined.map(image_cat)


# Function the collects both the GA/GQ data products of MODIS Aqua, renames the bands to readable versions, and joins the two collections into one.
def get_aqua(regiongeom, date_range):
    aqua_gq = (
        ee.ImageCollection("MODIS/061/MYD09GQ")
        .filterDate(date_range)
        .filterBounds(regiongeom)
    )
    aqua_ga = (
        ee.ImageCollection("MODIS/061/MYD09GA")
        .filterDate(date_range)
        .filterBounds(regiongeom)
    )
    return ee.ImageCollection(
        join_collections(dfo_bands_gq(aqua_gq), dfo_bands_ga(aqua_ga))
    )


# Function the collects both the GA/GQ data products of MODIS Terra, renames the bands to readable versions, and joins the two collections into one.
def get_terra(regiongeom, date_range):
    terra_gq = (
        ee.ImageCollection("MODIS/061/MOD09GQ")
        .filterDate(date_range)
        .filterBounds(regiongeom)
    )
    terra_ga = (
        ee.ImageCollection("MODIS/061/MOD09GA")
        .filterDate(date_range)
        .filterBounds(regiongeom)
    )
    return ee.ImageCollection(
        join_collections(dfo_bands_gq(terra_gq), dfo_bands_ga(terra_ga))
    )


# Pan sharpen the 500 images to the 250 resolution
def pan_sharpen(image):
    red_250m = image.select("red_250m")
    red_250m_safe = red_250m.where(red_250m.eq(0), 0.0000001)
    ratio = image.select("red_500m").divide(red_250m_safe)
    blue_ps = image.select("blue").divide(ratio)
    swir_ps = image.select("swir").divide(ratio)
    green_ps = image.select("green").divide(ratio)
    nir_500m_ps = image.select("nir_500m").divide(ratio)
    band5_ps = image.select("band5").divide(ratio)
    band6_ps = image.select("band6").divide(ratio)
    obs500_ps = image.select("obs500")
    return (
        image.select("red_250m", "nir_250m", "obs250", "state_1km")
        .addBands(
            [blue_ps, green_ps, swir_ps, nir_500m_ps, band5_ps, band6_ps, obs500_ps]
        )
        .set({"ratio_scale": ratio.projection().nominalScale()})
    )


# Calculate ratio of images
def b1b2_ratio(img):
    exp = "float(b('nir_250m') + 13.5) / float(b('red_250m') + 1081.1)"
    dfo_ratio = img.expression(exp)  # Band 1/Band 2 Ratio Threshold
    return img.addBands(dfo_ratio.select([0], ["b1b2_ratio"])).copyProperties(img)


# Calculate average
def calcmean(regiongeom, image):
    mean = image.reduceRegion(reducer=ee.Reducer.mean(), geometry=regiongeom, scale=250)
    return ee.Feature(None, mean)


# Calculate percentages
def calcpct(regiongeom, image):
    ptiles = image.reduceRegion(
        reducer=ee.Reducer.percentile([10, 20, 30, 40, 50, 60, 70, 80, 90]),
        geometry=regiongeom,
        scale=250,
    )
    return ee.Feature(None, ptiles)


# Creates an image based on MODIS QA information from the "state_1km" QA band.  This function has several bands including cloudy areas, cloud shadow, and snow/ice.
# QA Band information is available at: http://modis-sr.ltdri.org/guide/MOD09_UserGuide_v1_3.pdf
# Table 16: 1-kilometer State QA Descriptions (16-bit)
# cloud_state ==> 0: "clear", 1: "cloudy", 2: "mixed", 3: "not set"
# cloud_shadow ==> 0: "no", 1: "yes"
# ice_flag ==> 0: "no", 1: "yes"
# snow_flag ==> 0: "no snow", 1: "snow"
def get_qa_bits(image, start, end, new_name):
    # Compute the bits we need to extract.
    pattern = 0
    for i in range(start, end + 1):
        pattern += pow(2, i)
    return image.select([0], [new_name]).bitwiseAnd(pattern).rightShift(start)


def add_qa_bands(img):
    """Modified QA band addition with graceful fallback."""
    try:
        # Try to select state_1km band
        qa_band = img.select("state_1km")

        # Process QA bits if band exists
        cloud_state = get_qa_bits(qa_band, 0, 1, "cloud_state")
        cloud_shadow = get_qa_bits(qa_band, 2, 2, "cloud_shadow")
        ice_flag = get_qa_bits(qa_band, 12, 12, "ice_flag")
        snow_flag = get_qa_bits(qa_band, 15, 15, "snow_flag")

        return img.addBands([cloud_state, cloud_shadow, ice_flag, snow_flag])

    except ee.EEException:
        logger.warning("QA bands not available, skipping QA processing")
        # Return original image if QA processing fails
        return img


# Calculate water based on thresholds
def water_detection(modis_collection, thresh_b1b2, thresh_b1, thresh_b7):
    def water_flag(img):
        # Apply thresholds to each ratio/ band
        b1b2_ratio = ee.Image(img.select("b1b2_ratio"))
        b1b2_sliced = b1b2_ratio.lt(ee.Image.constant(thresh_b1b2))  # Band 1/Band 2
        b1_sliced = img.select(["red_250m"], ["b1_thresh"]).lt(
            ee.Image.constant(2027)
        )  # Band 1 Threshold
        b7_sliced = img.select(["swir"], ["b7_thresh"]).lt(
            ee.Image.constant(thresh_b7)
        )  # Band 7 Threshold

        # Add all the thresholds to one image and then sum()
        thresholds = b1b2_sliced.addBands(b1_sliced).addBands(b7_sliced)
        thresholds_count = thresholds.reduce(ee.Reducer.sum())

        # Apply water_flag threshold to final image
        water_flag = thresholds_count.gte(ee.Image.constant(3))
        return water_flag.copyProperties(img).set(
            "system:time_start", img.get("system:time_start")
        )

    # Apply the 'water_flag' function over the modis collection
    water_collection = modis_collection.map(water_flag)
    return water_collection.set(
        {
            "threshold_b1b2": round(thresh_b1b2, 3),
            "threshold_b7": round(thresh_b7, 2),
            "threshold_b1": round(thresh_b1, 2),
        }
    )


def process_polygon(polygon_id, shapefile, id_field, date_range):
    """Process a single polygon with proper dataframe creation and error handling."""
    try:
        logger.info(f"Processing polygon: {polygon_id}")
        regiongeom = shapefile.filterMetadata(id_field, "equals", polygon_id).geometry()

        # Get MODIS images
        aqua = get_aqua(regiongeom, date_range)
        terra = get_terra(regiongeom, date_range)

        # Add a limit to the number of images to process
        max_images = 1000  # Reduced to help with rate limiting
        aqua = aqua.limit(max_images)
        terra = terra.limit(max_images)

        # Pan-sharpen
        terra_sharp = terra.map(pan_sharpen)
        aqua_sharp = aqua.map(pan_sharpen)

        # Add band 1 and 2 ratio
        terra_ratio = terra_sharp.map(b1b2_ratio)
        aqua_ratio = aqua_sharp.map(b1b2_ratio)

        # Apply QA Band Extract
        terra_final = terra_ratio.map(add_qa_bands)
        aqua_final = aqua_ratio.map(add_qa_bands)

        # Merge the Terra and Aqua products with limit
        modis = ee.ImageCollection(
            terra_final.merge(aqua_final)
            .sort("system:time_start", True)
            .limit(max_images)
        )

        # Calculate statistics in chunks
        def process_statistics_chunk(collection, chunk_size=200):
            """Process statistics in smaller chunks to avoid the 5000 element limit."""
            results = []
            collection_size = collection.size().getInfo()

            for i in range(0, collection_size, chunk_size):
                chunk = collection.toList(chunk_size, i)
                chunk = ee.ImageCollection(chunk)

                # Calculate means and percentiles for this chunk
                bandmean = chunk.map(lambda img: calcmean(regiongeom, img))
                bandpct = chunk.map(lambda img: calcpct(regiongeom, img))

                # Convert to DataFrames
                try:
                    bandmean_df = fc_to_dataframe(bandmean)
                    bandpct_df = fc_to_dataframe(bandpct)

                    if not bandmean_df.empty and not bandpct_df.empty:
                        # Merge the DataFrames
                        chunk_df = pd.merge(
                            bandmean_df, bandpct_df, on="index", how="outer"
                        )
                        results.append(chunk_df)
                except Exception as e:
                    logger.warning(f"Error processing chunk {i}: {e}")
                    continue

            return pd.concat(results) if results else pd.DataFrame()

        # Process water detection
        thresh_b1b2 = 675
        thresh_b1 = 2027
        thresh_b7 = 675
        water = water_detection(modis, thresh_b1b2, thresh_b1, thresh_b7)

        # Calculate with lower bound threshold
        thresh_b1b2 = 540
        thresh_b1 = 1621.6
        thresh_b7 = 540
        water_lb = water_detection(modis, thresh_b1b2, thresh_b1, thresh_b7)

        # Calculate with upper bound threshold
        thresh_b1b2 = 810
        thresh_b1 = 2432.4
        thresh_b7 = 810
        water_ub = water_detection(modis, thresh_b1b2, thresh_b1, thresh_b7)

        # Process statistics in chunks
        main_stats_df = process_statistics_chunk(modis)
        water_stats_df = process_statistics_chunk(water)
        water_lb_stats_df = process_statistics_chunk(water_lb)
        water_ub_stats_df = process_statistics_chunk(water_ub)

        # Merge all DataFrames
        dfs_to_merge = [
            main_stats_df,
            water_stats_df,
            water_lb_stats_df,
            water_ub_stats_df,
        ]

        # Check if any DataFrame is empty
        if any(df.empty for df in dfs_to_merge):
            logger.warning(f"One or more empty DataFrames for polygon {polygon_id}")
            return pd.DataFrame()

        # Merge DataFrames
        final_df = reduce(
            lambda left, right: pd.merge(left, right, on="index", how="outer"),
            dfs_to_merge,
        )

        # Add polygon ID column
        final_df["polygon"] = str(polygon_id)

        logger.info(f"Successfully processed polygon {polygon_id}")
        return final_df

    except Exception as e:
        logger.error(f"Error processing polygon {polygon_id}: {e}")
        return pd.DataFrame()


def process_polygon_with_retry(
    polygon_id: str,
    date_range: ee.DateRange,
    shapefile: ee.FeatureCollection,
    id_field: str,
) -> Optional[pd.DataFrame]:
    """Process a single polygon with retry mechanism."""
    max_retries = 3
    retry_delay = 30  # Increased delay to help with rate limiting

    for attempt in range(max_retries):
        try:
            logger.info(f"Processing polygon {polygon_id} - Attempt {attempt + 1}")

            # Fixed: Pass all required arguments to process_polygon correctly
            result = process_polygon(
                polygon_id,
                shapefile,
                id_field,
                date_range
            )

            if result is not None and not result.empty:
                logger.info(f"Successfully processed polygon {polygon_id}")
                return result

        except ee.EEException as e:
            logger.warning(f"EE Exception for polygon {polygon_id}: {e}")
            if "rate" in str(e).lower() or "quota" in str(e).lower() or "429" in str(e):
                # Rate limiting - wait longer
                logger.info(f"Rate limit detected, waiting {retry_delay * 2} seconds")
                time.sleep(retry_delay * 2)
            else:
                time.sleep(retry_delay)
        except Exception as e:
            logger.error(f"Error processing polygon {polygon_id}: {e}", exc_info=True)
            if attempt == max_retries - 1:
                return None
            time.sleep(retry_delay)

    logger.error(f"Failed to process polygon {polygon_id} after {max_retries} attempts")
    return None


def save_results(batch_results: list, output_folder: str, failed_ids: list) -> None:
    """Save results and failed IDs to files in the specified folder."""
    try:
        if not batch_results:
            logger.warning("No results to save")
            return

        # Create timestamp for filenames
        timestamp = time.strftime("%Y%m%d_%H%M%S")

        # Ensure output folder exists
        os.makedirs(output_folder, exist_ok=True)

        # Define output paths
        results_file = os.path.join(output_folder, f"modis_results_{timestamp}.csv")
        failed_ids_file = os.path.join(
            output_folder, f"modis_failed_ids_{timestamp}.csv"
        )

        # Save main results
        final_df = pd.concat(batch_results, ignore_index=True)
        final_df.to_csv(results_file, index=False)
        logger.info(f"Saved results to {results_file}")

        # Save failed IDs if any
        if failed_ids:
            pd.DataFrame({"failed_id": failed_ids}).to_csv(failed_ids_file, index=False)
            logger.info(f"Saved {len(failed_ids)} failed IDs to {failed_ids_file}")

    except Exception as e:
        logger.error(f"Error saving results: {e}")
        raise


def process_polygons_in_batches(
    ids: List[str],
    date_range: ee.DateRange,
    shapefile: ee.FeatureCollection,
    config: Dict[str, Any],
):
    """Process polygons and save results to the specified output folder."""
    batch_results = []
    failed_ids = []

    # Reduced max_workers to help with rate limiting
    logger.info("Number of CPU cores:", cpu_count())
    max_workers = min(48, config["max_workers"])  # Cap at 2 workers to reduce rate limiting
    logger.info("Using workers: ", {max_workers})

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_id = {
            executor.submit(
                process_polygon_with_retry,
                polygon_id=pid,
                date_range=date_range,
                shapefile=shapefile,
                id_field=config["id_field"],
            ): pid
            for pid in ids
        }

        for idx, future in enumerate(tqdm(future_to_id, total=len(ids))):
            try:
                result = future.result()
                if result is not None and not result.empty:
                    batch_results.append(result)
                else:
                    failed_ids.append(future_to_id[future])

                # Add delay between batches to help with rate limiting
                if idx % 5 == 0 and idx > 0:
                    logger.info(f"Processed {idx} polygons, pausing briefly...")
                    time.sleep(10)

            except Exception as e:
                logger.error(
                    f"Error processing future for polygon {future_to_id[future]}: {e}"
                )
                failed_ids.append(future_to_id[future])

    # Save results using the output folder from config
    if batch_results or failed_ids:
        save_results(batch_results, config["output_path"], failed_ids)


# Initialize logger globally
logger = setup_logger("/content/drive/MyDrive/SAR_rgl/output_2/modis_processing.log")
logger.info("Starting main execution...")

# Load configuration
logger.info("Loading configuration...")
config = load_config("/content/drive/MyDrive/SAR_rgl/config_modis.yaml")
logger.info(f"Configuration loaded: {config}")

# Initialize Earth Engine
logger.info("Setting up Earth Engine...")
print("Starting main processing function...")
print("Setting up environment...")
shapefile, ids, dem, hand = setup_environment()

num_workers = min(cpu_count(), config["max_workers"])
print(f"Number of workers to be used: {num_workers}")
logger.info(f"Workers used: {num_workers}")
logger.info("Earth Engine setup completed")

# Set up date range
logger.info("Setting up date range...")
try:
    date_range = ee.DateRange(
        ee.Date(config["start_date"]), ee.Date(config["end_date"])
    )
    logger.info(f"Date range set: {config['start_date']} to {config['end_date']}")
except Exception as e:
    logger.error(f"Error setting up date range: {e}", exc_info=True)
    raise

# Process polygons
logger.info("Starting polygon processing...")
try:
    process_polygons_in_batches(
        ids=ids, date_range=date_range, shapefile=shapefile, config=config
    )
except Exception as e:
    logger.error(f"Error in process_polygons_in_batches: {e}", exc_info=True)
    raise

logger.info("Processing completed successfully")